# OpenStreetMapからPOI候補を取得する

## 1. 今回やること
OpenStreetMapから吉祥寺周辺のPOI候補を取得し、OSMタグを確認して、amenity=cafeのPOIを抽出します。他データとの比較は行いません。

© [OpenStreetMap contributors](https://www.openstreetmap.org/copyright)、データはODbL 1.0。
[実行手順](../docs/openstreetmap.md) / [検証記録](../docs/openstreetmap-validation.md)

## 2. 準備
Python / uvとOSMnx 2.1.1を使います。`uv sync --locked --group notebooks --group osm` で準備してください。APIキー・token・アカウントは不要です。public Overpassを使うため、小範囲・単発取得とし、保存済み応答を優先します。

In [ ]:
from geoai_open_lab.openstreetmap import (
    load_aoi, fetch_osm_pois, filter_osm_cafes, summarize_osm, save_osm_results,
)

## 3. 吉祥寺周辺のPOI候補を取得
amenity / shop / tourism / leisure / office / craft / healthcare / historicの8キーのOR条件です。公式POI一覧ではなく今回の実験用定義です。Pointは元座標、面はrepresentative_pointを用い、その点がAOI内（境界含む）のものを採用します。面がAOIと交差していても代表点が外なら除外されます。

In [ ]:
kichijoji = load_aoi("configs/aoi/kichijoji.toml")
places = fetch_osm_pois(bbox=kichijoji)

## 4. タグを確認
1つのcategory列ではなくkey=valueのタグで表現します。OSMの種類とIDを組み合わせて地物を識別します。OSMnxがgeometry化できるrelationには制約があります。

In [ ]:
places[["osm_type", "osm_id", "name", "amenity", "shop", "tourism", "leisure", "longitude", "latitude"]].head()

## 5. カフェを抽出
今回はOSMでamenity=cafeとタグ付けされたPOIです。Internet Cafe、コーヒー小売、restaurant、cafe=yesだけの一致は自動追加しません。

In [ ]:
cafes = filter_osm_cafes(places)
cafes[["osm_type", "osm_id", "name", "longitude", "latitude"]].head(10)

## 6. 結果と注意点
OSMは更新され続けます。元の取得時刻とbase timestampを記録し、再実行では保存済み応答を再処理します。POI候補数は実験用タグ・geometry対応・代表点の選択条件に依存し、現実の全店舗数ではありません。cafe=yesは今回の8キー条件で取得した候補内だけの集計です。

In [ ]:
summary = summarize_osm(places)
print(f"POI候補: {len(places)}件 / amenity=cafe: {len(cafes)}件")
summary["osm_types"], summary["geometry_types"]

In [ ]:
summary["related_tags"]

In [ ]:
result_dir = save_osm_results(places)